# 74: James Check Framework Deep Investigation

**Date:** 2026-01-21  
**Purpose:** Investigate backtesting results for Check's Buy The Dip + Exit frameworks

## Questions to Answer:

1. **Why did custom backtest inflate results?** (+473% vs +271% actual)
2. **Can we beat buy-and-hold?** Current answer: No (-86% with best strategy)
3. **Is the framework truly valuable?** Risk-adjusted returns vs absolute
4. **What about different parameter combinations?** Entry/exit threshold tuning
5. **How does it perform in bear markets?** Only tested 2023-2026 bull
6. **Trade-by-trade analysis:** When did it work/fail?

## Current Verified Results (VectorBT):

| Strategy | Return | Sharpe | Max DD | Trades |
|----------|--------|--------|--------|--------|
| Buy & Hold | +437.5% | ~1.0 | ? | - |
| LTH Distribution | +350.9% | 1.67 | -20.1% | 7 |
| 8-Metric 6/8 | +270.9% | 1.26 | -32.0% | 4 |
| 8-Metric 4/8 | +128.1% | 0.79 | -32.0% | 5 |

## Investigation Plan:

1. Load data and reproduce VectorBT results
2. Analyze custom backtest code to find bugs
3. Trade-by-trade breakdown
4. Parameter sensitivity analysis
5. Monte Carlo simulation
6. Compare to other Bitcoin strategies
7. Calculate buy-and-hold drawdown for fair comparison

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Try to import vectorbt
try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    print("✗ VectorBT not installed. Installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt
    print("✓ VectorBT installed and loaded")

# Plotting config
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("Setup complete!")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"

# Split dates
TRAIN_END = '2022-12-31'
TEST_START = '2023-01-01'

# Costs
FEES = 0.001  # 0.1%
SLIPPAGE = 0.001  # 0.1%

print(f"Data directory: {DATA_DIR}")
print(f"Train end: {TRAIN_END}")
print(f"Test start: {TEST_START}")

## 1. Data Loading

Load all required metrics for the Check framework.

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    """Load metric as time-indexed Series."""
    if source == "brk":
        path = DATA_DIR / f"{name}.parquet"
    elif source == "glassnode":
        path = GLASSNODE_DIR / f"{name}.parquet"
    else:
        return pd.Series(dtype=float)
    
    if not path.exists():
        return pd.Series(dtype=float)
    
    df = pd.read_parquet(path)
    
    # Normalize
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        df = df.set_index('time')['value']
        df = df.sort_index()
        return df
    
    return pd.Series(dtype=float)


def load_all_data() -> pd.DataFrame:
    """Load all metrics into aligned DataFrame."""
    print("Loading data...")
    
    metrics = {
        'price': ('price', 'brk'),
        'mvrv': ('mvrv', 'brk'),
        'mvrv_sth': ('mvrv_sth', 'brk'),
        'mvrv_lth': ('mvrv_lth', 'brk'),
        'sopr': ('sopr', 'brk'),
        'sopr_sth': ('sopr_sth', 'brk'),
        'sopr_lth': ('sopr_lth', 'brk'),
        'realized_profit': ('realized_profit', 'brk'),
        'realized_loss': ('realized_loss', 'brk'),
        'puell': ('puell_multiple', 'brk'),
        'price_200sma': ('price_200d_sma', 'brk'),
        'funding': ('funding_rate', 'glassnode'),
        'liq_long': ('liquidations_long', 'glassnode'),
        'liq_short': ('liquidations_short', 'glassnode'),
    }
    
    df_dict = {}
    for name, (metric, source) in metrics.items():
        series = load_metric(metric, source)
        if not series.empty:
            df_dict[name] = series
            print(f"  ✓ {name}")
        else:
            print(f"  ✗ {name} (missing)")
    
    # Combine all series
    df = pd.DataFrame(df_dict)
    
    # Forward fill missing values
    df = df.fillna(method='ffill')
    
    print(f"\nData loaded: {len(df)} days from {df.index[0].date()} to {df.index[-1].date()}")
    
    return df

# Load data
df = load_all_data()
df.head()

In [ ]:
# Split train/test
train_df = df[df.index <= TRAIN_END].copy()
test_df = df[df.index >= TEST_START].copy()

print(f"Train: {train_df.index[0].date()} to {train_df.index[-1].date()} ({len(train_df)} days)")
print(f"Test:  {test_df.index[0].date()} to {test_df.index[-1].date()} ({len(test_df)} days)")

## 2. Signal Generation

Generate entry and exit signals using Check's framework.

In [ ]:
def calculate_rolling_zscore(series: pd.Series, window: int) -> pd.Series:
    """Calculate rolling z-score using ONLY past data."""
    mean = series.rolling(window, min_periods=30).mean()
    std = series.rolling(window, min_periods=30).std()
    return (series - mean) / std


def generate_buy_the_dip_entries(df: pd.DataFrame) -> pd.Series:
    """
    Generate Buy The Dip entry signals.
    CRITICAL: Each row only uses data available UP TO that date.
    """
    # Condition 1: STH-MVRV < 1.0
    c1 = df['mvrv_sth'] < 1.0
    
    # Condition 2: STH-SOPR < 1.0
    c2 = df['sopr_sth'] < 1.0
    
    # Condition 3: RP/L Ratio < 1.0
    rplr = df['realized_profit'] / df['realized_loss']
    c3 = rplr < 1.0
    
    # Condition 4: Funding <= 0
    c4 = df['funding'] <= 0.0
    
    # Condition 5: Long Liq > Short Liq
    liq_ratio = df['liq_long'] / df['liq_short']
    c5 = liq_ratio > 1.0
    
    # Count conditions
    count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
    
    # Entry when 4+ conditions met
    entries = count >= 4
    
    return entries


def generate_lth_distribution_exits(df: pd.DataFrame) -> pd.Series:
    """Generate LTH Distribution exit signals."""
    exits = (df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)
    return exits


def generate_8metric_exits(df: pd.DataFrame, threshold: int = 6) -> pd.Series:
    """Generate 8-Metric exit signals."""
    # Calculate Z-scores using ONLY past data
    mvrv_z = calculate_rolling_zscore(df['mvrv'], 1460)
    mvrv_sth_z = calculate_rolling_zscore(df['mvrv_sth'], 365)
    sopr_z = calculate_rolling_zscore(df['sopr'], 365)
    sopr_sth_z = calculate_rolling_zscore(df['sopr_sth'], 365)
    puell_z = calculate_rolling_zscore(df['puell'], 365)
    
    # Mayer Multiple
    mayer = df['price'] / df['price_200sma']
    mayer_z = calculate_rolling_zscore(mayer, 365)
    
    # Funding Z
    funding_z = calculate_rolling_zscore(df['funding'], 365)
    
    # Count triggers
    count = (
        (mvrv_z > 1.5).astype(int) +
        (mvrv_sth_z > 1.25).astype(int) +
        (sopr_z > 1.5).astype(int) +
        (sopr_sth_z > 1.0).astype(int) +
        (mayer_z > 1.0).astype(int) +
        (puell_z > 1.5).astype(int) +
        (funding_z > 1.5).astype(int)
    )
    
    exits = count >= threshold
    
    return exits

# Generate signals for test period
test_entries = generate_buy_the_dip_entries(test_df)
test_lth_exits = generate_lth_distribution_exits(test_df)
test_8m_exits = generate_8metric_exits(test_df, threshold=6)
test_4m_exits = generate_8metric_exits(test_df, threshold=4)

print(f"Entry signals: {test_entries.sum()}")
print(f"LTH Distribution exits: {test_lth_exits.sum()}")
print(f"8-Metric 6/8 exits: {test_8m_exits.sum()}")
print(f"8-Metric 4/8 exits: {test_4m_exits.sum()}")

## 3. VectorBT Backtesting

Run backtests using VectorBT to reproduce verified results.

In [ ]:
def backtest_strategy(df: pd.DataFrame, entries: pd.Series, exits: pd.Series, name: str) -> dict:
    """Run backtest using VectorBT and return results."""
    pf = vbt.Portfolio.from_signals(
        close=df['price'],
        entries=entries,
        exits=exits,
        fees=FEES,
        slippage=SLIPPAGE,
        init_cash=10000,
        freq='1D'
    )
    
    return {
        'name': name,
        'portfolio': pf,
        'total_return': pf.total_return() * 100,
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown() * 100,
        'num_trades': pf.trades.count(),
        'win_rate': pf.trades.win_rate() * 100 if pf.trades.count() > 0 else 0,
    }

# Backtest strategies
results = {}

results['lth'] = backtest_strategy(test_df, test_entries, test_lth_exits, "LTH Distribution")
results['8m'] = backtest_strategy(test_df, test_entries, test_8m_exits, "8-Metric 6/8")
results['4m'] = backtest_strategy(test_df, test_entries, test_4m_exits, "8-Metric 4/8")

# Buy and hold
bh_return = (test_df['price'].iloc[-1] / test_df['price'].iloc[0] - 1) * 100

# Results table
results_df = pd.DataFrame([
    {
        'Strategy': r['name'],
        'Return': f"{r['total_return']:.1f}%",
        'Sharpe': f"{r['sharpe']:.2f}",
        'Max DD': f"{r['max_dd']:.1f}%",
        'Trades': int(r['num_trades']),
        'Win Rate': f"{r['win_rate']:.1f}%"
    }
    for r in results.values()
])

# Add buy & hold
results_df = pd.concat([
    results_df,
    pd.DataFrame([{
        'Strategy': 'Buy & Hold',
        'Return': f"{bh_return:.1f}%",
        'Sharpe': '~1.0',
        'Max DD': '?',
        'Trades': '-',
        'Win Rate': '-'
    }])
], ignore_index=True)

print("\n" + "="*80)
print("BACKTEST RESULTS (OOS 2023-2026)")
print("="*80)
print(results_df.to_string(index=False))

## 4. Trade-by-Trade Analysis

Detailed breakdown of each trade to understand when/why signals triggered.

In [ ]:
# LTH Distribution trades
print("LTH Distribution Trade Details:")
print("="*80)
print(results['lth']['portfolio'].trades.records_readable)

# Extract trade info
lth_trades = results['lth']['portfolio'].trades.records_readable
print(f"\nTotal trades: {len(lth_trades)}")
print(f"Winners: {(lth_trades['PnL'] > 0).sum()}")
print(f"Losers: {(lth_trades['PnL'] < 0).sum()}")

In [ ]:
# 8-Metric 6/8 trades
print("8-Metric 6/8 Trade Details:")
print("="*80)
print(results['8m']['portfolio'].trades.records_readable)

# Extract trade info
m8_trades = results['8m']['portfolio'].trades.records_readable
print(f"\nTotal trades: {len(m8_trades)}")
print(f"Winners: {(m8_trades['PnL'] > 0).sum()}")
print(f"Losers: {(m8_trades['PnL'] < 0).sum()}")

## 5. Equity Curves

Visualize portfolio value over time.

In [ ]:
# Plot equity curves
fig, ax = plt.subplots(figsize=(14, 7))

# Buy and hold baseline
bh_equity = (test_df['price'] / test_df['price'].iloc[0]) * 10000
ax.plot(test_df.index, bh_equity, label='Buy & Hold', linewidth=2, alpha=0.7, color='black', linestyle='--')

# Strategy equity curves
for key, res in results.items():
    equity = res['portfolio'].value()
    ax.plot(equity.index, equity.values, label=res['name'], linewidth=2, alpha=0.8)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('Equity Curves: Check Framework vs Buy & Hold (2023-2026)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Values:")
print(f"  Buy & Hold: ${bh_equity.iloc[-1]:,.0f}")
for key, res in results.items():
    final_val = res['portfolio'].value().iloc[-1]
    print(f"  {res['name']}: ${final_val:,.0f}")

## 6. Drawdown Analysis

Compare drawdowns to understand risk.

In [ ]:
# Plot drawdowns
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Buy & Hold drawdown
bh_dd = (bh_equity / bh_equity.cummax() - 1) * 100
axes[0].fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.3, color='black')
axes[0].plot(bh_dd.index, bh_dd.values, color='black', linewidth=1.5)
axes[0].set_title(f'Buy & Hold\nMax DD: {bh_dd.min():.1f}%', fontweight='bold')
axes[0].set_ylabel('Drawdown (%)')
axes[0].grid(True, alpha=0.3)

# Strategy drawdowns
for idx, (key, res) in enumerate(results.items(), 1):
    dd = res['portfolio'].drawdown() * 100
    axes[idx].fill_between(dd.index, dd.values, 0, alpha=0.3)
    axes[idx].plot(dd.index, dd.values, linewidth=1.5)
    axes[idx].set_title(f"{res['name']}\nMax DD: {res['max_dd']:.1f}%", fontweight='bold')
    axes[idx].set_ylabel('Drawdown (%)')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDrawdown Comparison:")
print(f"  Buy & Hold: {bh_dd.min():.1f}%")
for key, res in results.items():
    print(f"  {res['name']}: {res['max_dd']:.1f}%")

## 7. Signal Timeline Visualization

Plot when entry/exit signals triggered.

In [ ]:
# Plot signals on price chart
fig, ax = plt.subplots(figsize=(14, 7))

# Price
ax.plot(test_df.index, test_df['price'], label='BTC Price', color='black', linewidth=2, alpha=0.7)

# Entry signals
entry_dates = test_df.index[test_entries]
entry_prices = test_df.loc[entry_dates, 'price']
ax.scatter(entry_dates, entry_prices, color='green', marker='^', s=200, label='Buy The Dip Entry', zorder=5)

# LTH Distribution exits
lth_exit_dates = test_df.index[test_lth_exits]
lth_exit_prices = test_df.loc[lth_exit_dates, 'price']
ax.scatter(lth_exit_dates, lth_exit_prices, color='orange', marker='v', s=200, label='LTH Distribution Exit', zorder=5)

# 8-Metric exits
m8_exit_dates = test_df.index[test_8m_exits]
m8_exit_prices = test_df.loc[m8_exit_dates, 'price']
ax.scatter(m8_exit_dates, m8_exit_prices, color='red', marker='v', s=200, label='8-Metric 6/8 Exit', zorder=5)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('BTC Price ($)', fontsize=12)
ax.set_title('Check Framework Signals: Entry and Exit Timing (2023-2026)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"\nSignal Timing:")
print(f"\nEntry signals ({len(entry_dates)}):")
for date in entry_dates:
    price = test_df.loc[date, 'price']
    print(f"  {date.date()}: ${price:,.0f}")

print(f"\nLTH Distribution exits ({len(lth_exit_dates)}):")
for date in lth_exit_dates:
    price = test_df.loc[date, 'price']
    print(f"  {date.date()}: ${price:,.0f}")

print(f"\n8-Metric 6/8 exits ({len(m8_exit_dates)}):")
for date in m8_exit_dates:
    price = test_df.loc[date, 'price']
    print(f"  {date.date()}: ${price:,.0f}")

## 8. Parameter Sensitivity Analysis

Test different thresholds to see if we can improve results.

In [ ]:
# Test different entry thresholds (3/5, 4/5, 5/5)
entry_thresholds = [3, 4, 5]
exit_configs = [
    ('LTH Distribution', test_lth_exits),
    ('8-Metric 6/8', test_8m_exits),
]

sensitivity_results = []

for entry_thresh in entry_thresholds:
    # Generate entries with threshold
    c1 = test_df['mvrv_sth'] < 1.0
    c2 = test_df['sopr_sth'] < 1.0
    c3 = (test_df['realized_profit'] / test_df['realized_loss']) < 1.0
    c4 = test_df['funding'] <= 0.0
    c5 = (test_df['liq_long'] / test_df['liq_short']) > 1.0
    count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
    entries = count >= entry_thresh
    
    for exit_name, exits in exit_configs:
        res = backtest_strategy(test_df, entries, exits, f"Entry {entry_thresh}/5 + {exit_name}")
        sensitivity_results.append({
            'Entry Threshold': f"{entry_thresh}/5",
            'Exit Strategy': exit_name,
            'Return': f"{res['total_return']:.1f}%",
            'Sharpe': f"{res['sharpe']:.2f}",
            'Max DD': f"{res['max_dd']:.1f}%",
            'Trades': int(res['num_trades']),
        })

sensitivity_df = pd.DataFrame(sensitivity_results)
print("\n" + "="*80)
print("PARAMETER SENSITIVITY ANALYSIS")
print("="*80)
print(sensitivity_df.to_string(index=False))

## 9. Risk-Adjusted Return Comparison

Calculate Sharpe, Sortino, Calmar ratios.

In [ ]:
# Risk metrics comparison
risk_metrics = []

for key, res in results.items():
    pf = res['portfolio']
    risk_metrics.append({
        'Strategy': res['name'],
        'Total Return': f"{res['total_return']:.1f}%",
        'Annual Return': f"{(pf.total_return() * 100) / 3:.1f}%",  # 3 years
        'Sharpe': f"{res['sharpe']:.2f}",
        'Sortino': f"{pf.sortino_ratio():.2f}",
        'Calmar': f"{pf.calmar_ratio():.2f}",
        'Max DD': f"{res['max_dd']:.1f}%",
    })

risk_df = pd.DataFrame(risk_metrics)
print("\n" + "="*80)
print("RISK-ADJUSTED RETURN METRICS")
print("="*80)
print(risk_df.to_string(index=False))

print("\nInterpretation:")
print("  Sharpe > 1.0: Good risk-adjusted returns")
print("  Sortino: Like Sharpe but only penalizes downside volatility")
print("  Calmar: Annual return / Max drawdown")

## 10. Monte Carlo Simulation

Test robustness with bootstrap resampling.

In [ ]:
# Monte Carlo bootstrap - resample returns
def monte_carlo_bootstrap(returns: pd.Series, n_sims: int = 1000) -> pd.DataFrame:
    """Bootstrap resample returns to generate distribution."""
    results = []
    for _ in range(n_sims):
        # Resample with replacement
        sample = returns.sample(n=len(returns), replace=True)
        cum_return = (1 + sample).prod() - 1
        results.append(cum_return * 100)
    return pd.Series(results)

# Run Monte Carlo for LTH Distribution
lth_returns = results['lth']['portfolio'].returns()
lth_mc = monte_carlo_bootstrap(lth_returns, n_sims=1000)

# Plot distribution
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(lth_mc, bins=50, alpha=0.7, edgecolor='black')
ax.axvline(results['lth']['total_return'], color='red', linestyle='--', linewidth=2, label='Actual Return')
ax.axvline(bh_return, color='black', linestyle='--', linewidth=2, label='Buy & Hold')
ax.set_xlabel('Total Return (%)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Monte Carlo Bootstrap: LTH Distribution Return Distribution (1000 sims)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nMonte Carlo Results (LTH Distribution):")
print(f"  Mean: {lth_mc.mean():.1f}%")
print(f"  Median: {lth_mc.median():.1f}%")
print(f"  Std Dev: {lth_mc.std():.1f}%")
print(f"  5th percentile: {lth_mc.quantile(0.05):.1f}%")
print(f"  95th percentile: {lth_mc.quantile(0.95):.1f}%")
print(f"  % beating B&H: {(lth_mc > bh_return).mean() * 100:.1f}%")

## 11. Custom Backtest Bug Investigation

Load and compare custom backtest to understand the bug.

In [ ]:
# Load custom backtest results if available
custom_results_path = RESULTS_DIR / "framework_backtest_results.json"

if custom_results_path.exists():
    with open(custom_results_path, 'r') as f:
        custom_results = json.load(f)
    
    print("Custom Backtest Results:")
    print("="*80)
    
    for strat_name, strat_data in custom_results['results'].items():
        if 'test' in strat_data:
            test_data = strat_data['test']
            print(f"\n{strat_name}:")
            print(f"  Custom Return: {test_data.get('total_return', 'N/A')}")
            print(f"  Custom Trades: {test_data.get('num_trades', 'N/A')}")
    
    print("\n" + "="*80)
    print("DISCREPANCY ANALYSIS")
    print("="*80)
    print("\n8-Metric 6/8:")
    if 'eight_metric_high_risk' in custom_results['results']:
        custom_ret = custom_results['results']['eight_metric_high_risk']['test']['total_return']
        vbt_ret = results['8m']['total_return']
        diff = custom_ret - vbt_ret
        print(f"  Custom: {custom_ret:.1f}%")
        print(f"  VectorBT: {vbt_ret:.1f}%")
        print(f"  Difference: {diff:.1f}% (inflated by {diff/vbt_ret*100:.1f}%)")
    
    print("\nLTH Distribution:")
    if 'lth_distribution' in custom_results['results']:
        custom_ret = custom_results['results']['lth_distribution']['test']['total_return']
        vbt_ret = results['lth']['total_return']
        diff = custom_ret - vbt_ret
        print(f"  Custom: {custom_ret:.1f}%")
        print(f"  VectorBT: {vbt_ret:.1f}%")
        print(f"  Difference: {diff:.1f}% (variance: {abs(diff)/vbt_ret*100:.1f}%)")
else:
    print("Custom backtest results not found.")
    print(f"Expected path: {custom_results_path}")

## 12. Conclusions and Next Steps

Summary of findings and recommendations.

In [ ]:
print("="*80)
print("INVESTIGATION SUMMARY")
print("="*80)

print("\n1. VERIFIED RESULTS (VectorBT):")
print(f"   Buy & Hold: {bh_return:.1f}%")
for key, res in results.items():
    underperformance = bh_return - res['total_return']
    print(f"   {res['name']}: {res['total_return']:.1f}% (underperforms B&H by {underperformance:.1f}%)")

print("\n2. RISK-ADJUSTED PERFORMANCE:")
print(f"   Buy & Hold Max DD: {bh_dd.min():.1f}%")
for key, res in results.items():
    print(f"   {res['name']}: Sharpe {res['sharpe']:.2f}, Max DD {res['max_dd']:.1f}%")

print("\n3. KEY INSIGHTS:")
print("   ✓ LTH Distribution provides best risk-adjusted returns")
print("   ✓ All strategies underperform buy-and-hold in absolute terms")
print("   ✓ Framework is risk management tool, not alpha generator")
print(f"   ✓ Trade-off: Accept {bh_return - results['lth']['total_return']:.0f}% less profit for {bh_dd.min() - results['lth']['max_dd']:.0f}% less drawdown")

print("\n4. CUSTOM BACKTEST BUG:")
print("   ⚠️  8-Metric exit showed +473% (custom) vs +271% (VectorBT)")
print("   ⚠️  Likely causes: look-ahead bias, trade execution bugs, or cash management")
print("   ✓ LESSON: Always verify with VectorBT")

print("\n5. RECOMMENDATIONS:")
print("   • IF prioritizing Sharpe ratio → Use LTH Distribution exit")
print("   • IF prioritizing absolute returns → Stick with buy-and-hold")
print("   • Consider hybrid: B&H with smaller position + active trading")
print("   • Paper trade for 1 full cycle before live money")
print("   • Always use VectorBT for backtesting")

print("\n6. FURTHER INVESTIGATION:")
print("   • Test on full history (2009-2026) including bear markets")
print("   • Analyze why 2023-2026 underperformed (bull market specific?)")
print("   • Try position sizing (scale in/out) instead of all-in/all-out")
print("   • Combine with other timing models (e.g., trend following)")
print("   • Debug custom backtest to understand exact bug location")

print("\n" + "="*80)

## Next Notebook Ideas:

- **75_position_sizing.ipynb** - Test gradual scaling instead of binary signals
- **76_full_history_test.ipynb** - Backtest on complete 2009-2026 dataset
- **77_debug_custom_backtest.ipynb** - Step through custom backtest to find bug
- **78_hybrid_strategies.ipynb** - Combine B&H with active management